# Banking77: recover evaluation on two free T4 GPUs
This notebook restores the completed single-seed pilot and never retrains or publicly publishes.
Select T4 x2, Internet on, and attach HF_TOKEN through Kaggle Secrets. Never paste its value into code.
Teacher evaluation uses GPU 0; both trained students use GPU 1. Scores are checkpointed every sixteen
batches and private backups are downloaded/checksummed. Runtime or allocation is not guaranteed.
Use Save Version → Save & Run All (Commit) for a server-owned run. All environments/model caches
stay under /kaggle/temp. After evaluation, restore the final private snapshot in Colab to publish
using Colab Secrets. See docs/EVALUATION_RECOVERY.md. No paid compute is provisioned.

For v0.1 the explicit release plan requires batch 1; batch-32 scaling remains future work.


In [ ]:
import os
import pathlib
import subprocess
import sys

source = pathlib.Path("/kaggle/working/Banking77EvaluationSource")
if not source.exists():
    subprocess.run(["git", "clone", "https://github.com/jatinmallarapu/Banking77Distillation.git", str(source)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.13.9"], check=True)
subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "/kaggle/temp/b77-eval-venv"], check=True)
eval_python = "/kaggle/temp/b77-eval-venv/bin/python"
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", eval_python, "-e", str(source) + "[train,teacher-eval,dev]"], check=True)
eval_env = dict(os.environ, PATH="/kaggle/temp/b77-eval-venv/bin:" + os.environ["PATH"], HF_HOME="/kaggle/temp/huggingface")
eval_env.pop("HF_TOKEN", None)
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True))


In [ ]:
import json
import shutil
from kaggle_secrets import UserSecretsClient

recovered = pathlib.Path("/kaggle/working/Banking77EvaluationRecovered")
if recovered.exists():
    raise RuntimeError("Preserve the existing recovery directory; do not restore over it")
result = subprocess.run(
    [eval_python, "-m", "b77.backup", "restore", "--token-stdin", "--destination", str(recovered)],
    cwd=source, env=eval_env, text=True, capture_output=True,
    input=UserSecretsClient().get_secret("HF_TOKEN"),
)
print(result.stdout, result.stderr[-2000:], flush=True)
result.check_returncode()
plan_path = recovered / "configs/release_plan.json"
plan = json.loads(plan_path.read_text())
plan["batch_sizes"] = [1]
plan_path.write_text(json.dumps(plan, indent=2) + "\n")
# Preserve recovered configs/weights; use the current tested evaluation and backup source.
for folder in ["src", "scripts"]:
    shutil.copytree(source / folder, recovered / folder, dirs_exist_ok=True)
process = subprocess.Popen(
    [eval_python, "-m", "b77.resume_evaluation", "--token-stdin", "--max-hours", "10"],
    cwd=recovered, env=eval_env, text=True, stdin=subprocess.PIPE,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
)
process.stdin.write(UserSecretsClient().get_secret("HF_TOKEN"))
process.stdin.close()
for line in process.stdout:
    print(line, end="", flush=True)
code = process.wait()
if code:
    raise RuntimeError(f"Evaluation stopped with code {code}; preserve the latest private backup")
print("EVALUATION COMPLETE; restore in Colab for public publishing", flush=True)
